## Baseline dataset ##

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

data1 = pd.read_csv(r'datasets\MachineLearningCVE\Monday-WorkingHours.pcap_ISCX.csv')
data2 = pd.read_csv(r'datasets\MachineLearningCVE\Tuesday-WorkingHours.pcap_ISCX.csv')
data3 = pd.read_csv(r'datasets\MachineLearningCVE\Wednesday-workingHours.pcap_ISCX.csv')
data4 = pd.read_csv(r'datasets\MachineLearningCVE\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv')
data5 = pd.read_csv(r'datasets\MachineLearningCVE\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv')
data6 = pd.read_csv(r'datasets\MachineLearningCVE\Friday-WorkingHours-Morning.pcap_ISCX.csv')
data7 = pd.read_csv(r'datasets\MachineLearningCVE\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')
data8 = pd.read_csv(r'datasets\MachineLearningCVE\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')

data_list = [data1, data2, data3, data4, data5, data6, data7, data8]
for df in data_list:
    df.columns = df.columns.str.strip()
data = pd.concat(data_list)
data["Label"] = data["Label"].str.strip()   # Standardize labels, remove whitespace
rows, cols = data.shape

## Preprocessing ##

In [3]:
# Duplicates check
dupes = data[data.duplicated()]
print(f"Number of duplicates: {len(dupes)}")
data.drop_duplicates(inplace=True)
after_dupes = data.shape
print(f"Rows & columns after dropping duplicate values: {after_dupes}")

Number of duplicates: 308381
Rows & columns after dropping duplicate values: (2522362, 79)


In [4]:
# Missing values check
# missing = data.isnull().sum().sum()
# print(missing)

audit = pd.DataFrame({
    "Null": data.isnull().sum(),
    "+inf": (data == np.inf).sum(),
    "-inf": (data == -np.inf).sum(),
    "zeros": (data == 0).sum()
})

affected_columns = audit[
    (audit["Null"] > 0) |
    (audit["+inf"] > 0) |
    (audit["-inf"] > 0)
].index
print("Missing or corrupted values:\n")
print(audit.loc[affected_columns])

data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True)

audit_after = pd.DataFrame({
    "Null": data.isnull().sum(),
    "+inf": (data == np.inf).sum(),
    "-inf": (data == -np.inf).sum(),
    "zeros": (data == 0).sum()
})
print("\nAfter dropping missing or corrupt values:\n")
print(audit_after.loc[affected_columns])

Missing or corrupted values:

                Null  +inf  -inf   zeros
Flow Bytes/s     353  1211     0  277504
Flow Packets/s     0  1564     0       0

After dropping missing or corrupt values:

                Null  +inf  -inf   zeros
Flow Bytes/s       0     0     0  277504
Flow Packets/s     0     0     0       0


In [5]:
# Feature selection: based on previous work

selected_features = [
    'Total Fwd Packets', 'Total Length of Fwd Packets',
    'Fwd Packet Length Max', 'Fwd Packet Length Mean',
    'Bwd Packet Length Max', 'Flow Packets/s',
    'Fwd IAT Std', 'Fwd IAT Min', 'Fwd Header Length',
    'Bwd Header Length', 'Max Packet Length',
    'Packet Length Mean', 'Packet Length Variance',
    'Init_Win_bytes_forward', 'Init_Win_bytes_backward'
    # Destination Port intentionally excluded - not sure if I want it yet?
]

X = data[selected_features]
y = data['Label']

baseline = X.copy()
baseline['Label'] = y.values

baseline.to_csv('outputs/processed/baseline.csv', index=False)
print(f"Saved baseline: {baseline.shape}")

Saved baseline: (2520798, 16)
